In [1]:
import math
import time
from pathlib import Path
import numpy as np
from numba import cuda

ROOT = Path.cwd()
if not (ROOT / 'mc_temperature_snapshots').exists():
    ROOT = Path('/home/senji77/akhyar/research/notebooks/ANSW')

OUTPUT = ROOT / 'asd_replica_profiles'
OUTPUT.mkdir(parents=True, exist_ok=True)
SNAPSHOTS = ROOT / 'mc_temperature_snapshots'
TEMPERATURES = [0.1, 0.3, 0.5, 1.0]
Q_TARGETS = np.array([(np.pi/2, 0.0),(np.pi/2, np.pi/4)],dtype=np.float64)
N_REPLICAS_ASD = 16
REPLICAS_PER_BATCH = 4
N_SAMPLES_ASD = 512
DT_ASD = 0.025
SAMPLE_EVERY_ASD = 5
BLOCK_ASD = (8,8,1)
J_asd = 1.0
q0_asd = np.pi/4
D_asd = J_asd*np.tan(q0_asd)
J2_asd = J_asd/(4*np.cos(q0_asd))
Bz_asd = 0.0

print('CUDA available:',cuda.is_available())
assert cuda.is_available(), 'This notebook requires NVIDIA CUDA'
print('Replica spectra output:',OUTPUT)

CUDA available: True
Replica spectra output: /home/senji77/akhyar/research/notebooks/ANSW/asd_replica_profiles


In [2]:
def target_tag(t):
    return f'{t:.1f}'.replace('.','p')


def load_independent_spins(model,t):
    if np.isclose(t,0.1):
        if model == 'dmi':
            candidates=[ROOT/'production_dmi_x'/'spins_dmi_latest.npy']
        else:
            candidates=[ROOT/'spins_frustration_latest.npy',
                        ROOT/'production_frustration'/'spins_frustration_latest.npy']
        path=next((p for p in candidates if p.exists()),None)
        assert path is not None, f'Missing T=0.1 spins for {model}: {candidates}'
        snapshots=np.load(path,mmap_mode='r')
        assert snapshots.ndim==4
        available=snapshots.shape[0]
    else:
        tag=target_tag(t)
        path=SNAPSHOTS/f'spins_{model}_T{tag}.npy'
        snapshots=np.load(path,mmap_mode='r')
        assert snapshots.ndim==5
        available=snapshots.shape[1]
        with np.load(SNAPSHOTS/f'mc_checks_{model}_T{tag}.npz') as d:
            assert np.isclose(float(d['D']),D_asd)
            assert np.isclose(float(d['J2']),J2_asd)
            assert np.isclose(float(d['B_z']),Bz_asd)
    assert N_REPLICAS_ASD <= available, f'{model} T={t}: only {available} independent MC replicas'
    rng=np.random.default_rng(20261115+int(t*1000)+(0 if model=='dmi' else 10000))
    replica_ids=rng.choice(available,size=N_REPLICAS_ASD,replace=False)
    if snapshots.ndim==4:
        selection=np.asarray(snapshots[replica_ids],dtype=np.float32)
    else:
        snapshot_ids=rng.integers(0,snapshots.shape[0],size=N_REPLICAS_ASD)
        selection=np.stack([snapshots[si,ri] for si,ri in zip(snapshot_ids,replica_ids)]).astype(np.float32)
    assert selection.shape[-1]==3
    assert selection.shape[1]==selection.shape[2]==64
    return np.ascontiguousarray(selection),replica_ids

In [3]:
@cuda.jit
def rhs_gpu(spins, derivatives, model_id, J, J2, D, Bz):
    x, y, r = cuda.grid(3)

    nx = spins.shape[1]
    ny = spins.shape[2]

    if r >= spins.shape[0] or x >= nx or y >= ny:
        return

    xp = (x + 1) % nx
    xm = (x - 1 + nx) % nx
    yp = (y + 1) % ny
    ym = (y - 1 + ny) % ny

    sx = spins[r, x, y, 0]
    sy = spins[r, x, y, 1]
    sz = spins[r, x, y, 2]

    hx = J * (
        spins[r, xp, y, 0] + spins[r, xm, y, 0]
        + spins[r, x, yp, 0] + spins[r, x, ym, 0]
    )

    hy = J * (
        spins[r, xp, y, 1] + spins[r, xm, y, 1]
        + spins[r, x, yp, 1] + spins[r, x, ym, 1]
    )

    hz = J * (
        spins[r, xp, y, 2] + spins[r, xm, y, 2]
        + spins[r, x, yp, 2] + spins[r, x, ym, 2]
    ) + Bz

    if model_id == 0:
        hy += D * (spins[r, xp, y, 2] - spins[r, xm, y, 2])
        hz -= D * (spins[r, xp, y, 1] - spins[r, xm, y, 1])
    else:
        xpp = (x + 2) % nx
        xmm = (x - 2 + nx) % nx

        hx -= J2 * (spins[r, xpp, y, 0] + spins[r, xmm, y, 0])
        hy -= J2 * (spins[r, xpp, y, 1] + spins[r, xmm, y, 1])
        hz -= J2 * (spins[r, xpp, y, 2] + spins[r, xmm, y, 2])

    derivatives[r, x, y, 0] = -(sy * hz - sz * hy)
    derivatives[r, x, y, 1] = -(sz * hx - sx * hz)
    derivatives[r, x, y, 2] = -(sx * hy - sy * hx)


@cuda.jit
def rk4_stage_gpu(spins, derivative, output, factor):
    x, y, r = cuda.grid(3)

    if r >= spins.shape[0] or x >= spins.shape[1] or y >= spins.shape[2]:
        return

    ax = spins[r, x, y, 0] + factor * derivative[r, x, y, 0]
    ay = spins[r, x, y, 1] + factor * derivative[r, x, y, 1]
    az = spins[r, x, y, 2] + factor * derivative[r, x, y, 2]

    inv = 1.0 / math.sqrt(ax * ax + ay * ay + az * az)

    output[r, x, y, 0] = ax * inv
    output[r, x, y, 1] = ay * inv
    output[r, x, y, 2] = az * inv


@cuda.jit
def rk4_finish_gpu(spins, k1, k2, k3, k4, dt):
    x, y, r = cuda.grid(3)

    if r >= spins.shape[0] or x >= spins.shape[1] or y >= spins.shape[2]:
        return

    factor = dt / 6.0

    ax = spins[r, x, y, 0] + factor * (
        k1[r, x, y, 0] + 2.0 * k2[r, x, y, 0]
        + 2.0 * k3[r, x, y, 0] + k4[r, x, y, 0]
    )

    ay = spins[r, x, y, 1] + factor * (
        k1[r, x, y, 1] + 2.0 * k2[r, x, y, 1]
        + 2.0 * k3[r, x, y, 1] + k4[r, x, y, 1]
    )

    az = spins[r, x, y, 2] + factor * (
        k1[r, x, y, 2] + 2.0 * k2[r, x, y, 2]
        + 2.0 * k3[r, x, y, 2] + k4[r, x, y, 2]
    )

    inv = 1.0 / math.sqrt(ax * ax + ay * ay + az * az)

    spins[r, x, y, 0] = ax * inv
    spins[r, x, y, 1] = ay * inv
    spins[r, x, y, 2] = az * inv

In [4]:
def energy_per_spin(spins, model):
    xp = np.roll(spins, -1, axis=1)
    yp = np.roll(spins, -1, axis=2)

    energy = -J_asd * np.sum(spins * (xp + yp), axis=(1, 2, 3), dtype=np.float64)

    if model == "dmi":
        energy += D_asd * np.sum(
            spins[..., 2] * xp[..., 1] - spins[..., 1] * xp[..., 2],
            axis=(1, 2), dtype=np.float64
        )
    else:
        xpp = np.roll(spins, -2, axis=1)
        energy += J2_asd * np.sum(spins * xpp, axis=(1, 2, 3), dtype=np.float64)

    energy -= Bz_asd * np.sum(spins[..., 2], axis=(1, 2), dtype=np.float64)

    return energy / (spins.shape[1] * spins.shape[2])


def rk4_step_cuda(d_spins, arrays, grid, model_id):
    d_stage, k1, k2, k3, k4 = arrays

    args = (model_id, np.float32(J_asd), np.float32(J2_asd),
            np.float32(D_asd), np.float32(Bz_asd))

    rhs_gpu[grid, BLOCK_ASD](d_spins, k1, *args)
    rk4_stage_gpu[grid, BLOCK_ASD](d_spins, k1, d_stage, np.float32(DT_ASD / 2))
    rhs_gpu[grid, BLOCK_ASD](d_stage, k2, *args)
    rk4_stage_gpu[grid, BLOCK_ASD](d_spins, k2, d_stage, np.float32(DT_ASD / 2))
    rhs_gpu[grid, BLOCK_ASD](d_stage, k3, *args)
    rk4_stage_gpu[grid, BLOCK_ASD](d_spins, k3, d_stage, np.float32(DT_ASD))
    rhs_gpu[grid, BLOCK_ASD](d_stage, k4, *args)
    rk4_finish_gpu[grid, BLOCK_ASD](d_spins, k1, k2, k3, k4, np.float32(DT_ASD))

In [5]:
def run_asd(model, starting_spins):
    model_id = 0 if model == "dmi" else 1
    nrep, nx, ny, _ = starting_spins.shape

    grid = (
        (nx + BLOCK_ASD[0] - 1) // BLOCK_ASD[0],
        (ny + BLOCK_ASD[1] - 1) // BLOCK_ASD[1],
        nrep
    )

    d_spins = cuda.to_device(np.ascontiguousarray(starting_spins, dtype=np.float32))
    arrays = [cuda.device_array_like(d_spins) for _ in range(5)]

    fourier_time = np.empty(
        (nrep, N_SAMPLES_ASD, nx, ny, 3),
        dtype=np.complex64
    )

    t_diagnostics = []
    e_diagnostics = []
    initial_energy = energy_per_spin(starting_spins, model)
    start = time.perf_counter()

    for sample in range(N_SAMPLES_ASD):
        if sample > 0:
            for _ in range(SAMPLE_EVERY_ASD):
                rk4_step_cuda(d_spins, arrays, grid, model_id)

        current = d_spins.copy_to_host()
        fourier_time[:, sample] = (
            np.fft.fft2(current, axes=(1, 2)) / np.sqrt(nx * ny)
        ).astype(np.complex64)

        if sample % 64 == 0 or sample == N_SAMPLES_ASD - 1:
            t_diagnostics.append(sample * DT_ASD * SAMPLE_EVERY_ASD)
            e_diagnostics.append(energy_per_spin(current, model))

        if (sample + 1) % 128 == 0 or sample + 1 == N_SAMPLES_ASD:
            print(f"{model}: {sample + 1}/{N_SAMPLES_ASD} samples", flush=True)

    final_spins = d_spins.copy_to_host()
    final_energy = energy_per_spin(final_spins, model)
    drift = np.max(
        np.abs(final_energy - initial_energy)
        / np.maximum(np.abs(initial_energy), 1e-10)
    )
    norm_error = np.max(np.abs(np.linalg.norm(final_spins, axis=-1) - 1))

    print(model, "max relative energy drift:", drift)
    print(model, "max spin norm error:", norm_error)
    print(model, "elapsed seconds:", time.perf_counter() - start)

    if drift > 5e-3:
        raise RuntimeError(f"Energy drift too large for {model}: {drift}")

    return fourier_time, np.array(t_diagnostics), np.array(e_diagnostics)

In [6]:
def replica_spectrum_at_q(fourier_time):
    nrep,nt,nx,ny,_=fourier_time.shape
    window=np.hanning(nt).astype(np.float32)
    dt_sample=DT_ASD*SAMPLE_EVERY_ASD
    factor=dt_sample/(nt*np.mean(window**2))
    freq=2*np.pi*np.fft.fftfreq(nt,dt_sample)
    positive=freq>=0
    qx=2*np.pi*np.fft.fftshift(np.fft.fftfreq(nx))
    qy=2*np.pi*np.fft.fftshift(np.fft.fftfreq(ny))
    result=np.empty((nrep,np.count_nonzero(positive),len(Q_TARGETS)),dtype=np.float32)
    measured=np.empty_like(Q_TARGETS)
    for iq,(target_x,target_y) in enumerate(Q_TARGETS):
        shift_x=int(np.argmin(abs(qx-target_x)))
        shift_y=int(np.argmin(abs(qy-target_y)))
        ix=(shift_x+nx//2)%nx
        iy=(shift_y+ny//2)%ny
        series=fourier_time[:,:,ix,iy,:].astype(np.complex64)
        series-=series.mean(axis=1,keepdims=True)
        series*=window[None,:,None]
        transform=np.fft.fft(series,axis=1)[:,positive,:]
        result[:,:,iq]=np.sum(np.abs(transform)**2,axis=-1)*factor
        measured[iq]=[qx[shift_x],qy[shift_y]]
    return freq[positive],result,measured

In [7]:
for t in TEMPERATURES:
    label=target_tag(t)
    spectra={}
    diagnostics={}
    for model in ['dmi','frustration']:
        spins,replica_ids=load_independent_spins(model,t)
        chunks=[]
        energy_drift=[]
        for start in range(0,N_REPLICAS_ASD,REPLICAS_PER_BATCH):
            batch=spins[start:start+REPLICAS_PER_BATCH]
            fourier,times,energies=run_asd(model,batch)
            omega,single_spectra,q_grid=replica_spectrum_at_q(fourier)
            chunks.append(single_spectra)
            energy_drift.extend(np.max(np.abs(energies-energies[0])/
                       np.maximum(abs(energies[0]),1e-10),axis=0))
            del fourier
            print(f'T={t:.1f} {model}: {min(start+REPLICAS_PER_BATCH,N_REPLICAS_ASD)}/{N_REPLICAS_ASD} replicas',flush=True)
        spectra[model]=np.concatenate(chunks,axis=0)
        diagnostics[model]=(replica_ids,np.asarray(energy_drift))
    file=OUTPUT/f'asd_replica_profiles_T{label}.npz'
    np.savez_compressed(file,
        temperature=t,omega=omega,q_targets=q_grid,
        S_dmi_replica=spectra['dmi'],
        S_frustration_replica=spectra['frustration'],
        replica_ids_dmi=diagnostics['dmi'][0],
        replica_ids_frustration=diagnostics['frustration'][0],
        relative_energy_drift_dmi=diagnostics['dmi'][1],
        relative_energy_drift_frustration=diagnostics['frustration'][1],
        n_replicas=N_REPLICAS_ASD,dt=DT_ASD,sample_every=SAMPLE_EVERY_ASD,
        J=J_asd,D=D_asd,J2=J2_asd,B_z=Bz_asd)
    print('Saved:',file)

dmi: 128/512 samples
dmi: 256/512 samples
dmi: 384/512 samples
dmi: 512/512 samples
dmi max relative energy drift: 2.2683945278244666e-05
dmi max spin norm error: 5.9604645e-08
dmi elapsed seconds: 2.197757867979817
T=0.1 dmi: 4/16 replicas
dmi: 128/512 samples
dmi: 256/512 samples
dmi: 384/512 samples
dmi: 512/512 samples
dmi max relative energy drift: 2.3558709250094036e-05
dmi max spin norm error: 5.9604645e-08
dmi elapsed seconds: 1.2878286060295068
T=0.1 dmi: 8/16 replicas
dmi: 128/512 samples
dmi: 256/512 samples
dmi: 384/512 samples
dmi: 512/512 samples
dmi max relative energy drift: 2.2750501840212432e-05
dmi max spin norm error: 5.9604645e-08
dmi elapsed seconds: 1.2054032509913668
T=0.1 dmi: 12/16 replicas
dmi: 128/512 samples
dmi: 256/512 samples
dmi: 384/512 samples
dmi: 512/512 samples
dmi max relative energy drift: 2.3313045066295983e-05
dmi max spin norm error: 5.9604645e-08
dmi elapsed seconds: 1.2185283669969067
T=0.1 dmi: 16/16 replicas
frustration: 128/512 samples
fr

In [8]:
for t in TEMPERATURES:
    file=OUTPUT/f'asd_replica_profiles_T{target_tag(t)}.npz'
    with np.load(file) as d:
        print('T=',float(d['temperature']),
              'shape=',d['S_dmi_replica'].shape,
              'DMI-x drift=',float(np.max(d['relative_energy_drift_dmi'])),
              'frustration drift=',float(np.max(d['relative_energy_drift_frustration'])))

T= 0.1 shape= (16, 256, 2) DMI-x drift= 2.3558709250094036e-05 frustration drift= 6.631864309183994e-06
T= 0.3 shape= (16, 256, 2) DMI-x drift= 5.1581551738025045e-05 frustration drift= 1.4323956052387484e-05
T= 0.5 shape= (16, 256, 2) DMI-x drift= 6.164572929863827e-05 frustration drift= 1.5591988344101226e-05
T= 1.0 shape= (16, 256, 2) DMI-x drift= 4.5348447632303254e-05 frustration drift= 1.432925339444787e-05
